#2.1 Load and inspect

In [ ]:
import pandas as pd
from scipy import stats
import numpy as np
df = pd.read_csv("DATASET.csv")

print(df.head())
print(df.info())
print(df.describe())

   event_id  session_id  user_id  variation platform  \
0  63527610     6391574   762832          2  Android   
1  12446736     6391574   762832          2  Android   
2  90232698     6391574   762832          2  Android   
3  74183469     9246026   762832          2  Android   
4  16360628     9246026   762832          2  Android   

                  datetime_event       event_type final_order_status  shop_id  
0  2024-11-30 16:23:46.391133804    entry_to_shop         successful   8531.0  
1  2024-11-30 16:26:16.875713965       order_paid         successful   8531.0  
2  2024-11-30 16:50:44.827088753   order_finished         successful   8531.0  
3  2024-11-29 13:21:28.729203894  reload_the_page                NaN   6186.0  
4  2024-11-29 13:21:35.213637901    entry_to_shop                NaN   6186.0  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 326921 entries, 0 to 326920
Data columns (total 9 columns):
 #   Column              Non-Null Count   Dtype  
---  ------            

#2.2 Create a session-level conversion flag

We want one row per (session_id, variation, platform) with:

converted = 1 if that session ever had a successful order,

0 otherwise.

In [ ]:
df = df.dropna()
display(df.value_counts())


,,,,,,,,,count
event_id,session_id,user_id,variation,platform,datetime_event,event_type,final_order_status,shop_id,
99999925,1153796,713665,2,iOS,2024-11-25 16:09:55.480439190,order_finished,successful,3869.0,1
10000878,6911681,657332,2,Android,2024-11-26 15:53:55.391118540,entry_to_shop,successful,5985.0,1
10001672,7379429,811359,2,Android,2024-11-27 17:45:43.940947976,order_paid,cancelled,1805.0,1
10002700,1987878,542747,1,iOS,2024-11-28 19:49:34.382465547,entry_to_shop,successful,4595.0,1
10003447,6535301,762927,1,Android,2024-11-29 14:22:25.852118016,reload_the_page,successful,4842.0,1
...,...,...,...,...,...,...,...,...,...
10007183,3785281,344722,2,Android,2024-11-29 20:26:43.847432581,order_finished,cancelled,4869.0,1
10006915,7120641,339961,1,iOS,2024-11-27 18:13:47.021868417,order_paid,cancelled,3873.0,1
10006132,2090231,367496,2,Android,2024-11-30 14:26:12.332668682,order_paid,successful,9961.0,1


#2.3 Map numeric variation to readable group names (A/B)

In [ ]:


# Parse datetime
df["datetime_event"] = pd.to_datetime(df["datetime_event"])

print(df.head())
print(df.dtypes)

# Check variations present
print("Variations:", df["variation"].unique())

# Check what final_order_status values exist
print(df["final_order_status"].value_counts(dropna=False).head(20))

# Check event types (for funnel/extra analysis)
print(df["event_type"].value_counts().head(20))


   event_id  session_id  user_id  variation platform  \
0  63527610     6391574   762832          2  Android   
1  12446736     6391574   762832          2  Android   
2  90232698     6391574   762832          2  Android   
6  79159985     9929810   762832          2  Android   
7  53691402     9929810   762832          2  Android   

                 datetime_event      event_type final_order_status  shop_id  
0 2024-11-30 16:23:46.391133804   entry_to_shop         successful   8531.0  
1 2024-11-30 16:26:16.875713965      order_paid         successful   8531.0  
2 2024-11-30 16:50:44.827088753  order_finished         successful   8531.0  
6 2024-11-28 17:42:49.168897336   entry_to_shop         successful   9082.0  
7 2024-11-28 17:44:13.997949301      order_paid         successful   9082.0  
event_id                       int64
session_id                     int64
user_id                        int64
variation                      int64
platform                      object
datetime_e

#2.4 Overall A/B test on conversion rate

In [ ]:
# 1. Flag successful order events
success_statuses = ["successful"]  # <-- change to match your data
df["is_success_event"] = df["final_order_status"].isin(success_statuses).astype(int)

# 2. Aggregate to session level
session_df = (
    df
    .groupby(["session_id", "variation", "platform"], as_index=False)
    .agg(
        converted=("is_success_event", "max")  # 1 if any success in that session
    )
)

print(session_df.head())
print(session_df["variation"].value_counts())


   session_id  variation platform  converted
0     1000058          2      iOS          1
1     1000247          1  Android          1
2     1000733          1      iOS          1
3     1000803          1  Android          0
4     1000867          1  Android          0
variation
1    34140
2    18278
Name: count, dtype: int64


#Two-proportion z-test


In [ ]:
# Identify control and treatment based on variation values
var_values = sorted(session_df["variation"].unique())
control_var = var_values[0]
treatment_var = var_values[1] if len(var_values) > 1 else var_values[0]

session_df["group"] = np.where(
    session_df["variation"] == control_var, "control", "treatment"
)

print(session_df["group"].value_counts())


group
control      34140
treatment    18278
Name: count, dtype: int64


#95% confidence interval

In [ ]:
control = session_df[session_df["group"] == "control"]
treatment = session_df[session_df["group"] == "treatment"]

n_control = len(control)
n_treat = len(treatment)

conv_control = control["converted"].mean()
conv_treat = treatment["converted"].mean()

print(f"Control sessions: {n_control}, conversion rate: {conv_control:.4f}")
print(f"Treatment sessions: {n_treat}, conversion rate: {conv_treat:.4f}")

# Lift
absolute_lift = conv_treat - conv_control
relative_lift = absolute_lift / conv_control if conv_control > 0 else np.nan

print(f"Absolute lift (T - C): {absolute_lift:.4f}")
print(f"Relative lift: {relative_lift * 100:.2f}%")


Control sessions: 34140, conversion rate: 0.8395
Treatment sessions: 18278, conversion rate: 0.8861
Absolute lift (T - C): 0.0466
Relative lift: 5.55%


#3. Optional: Break down by platform

In [ ]:
# successes
success_control = control["converted"].sum()
success_treat = treatment["converted"].sum()

# pooled proportion
p_pool = (success_control + success_treat) / (n_control + n_treat)

# standard error
se = np.sqrt(p_pool * (1 - p_pool) * (1/n_control + 1/n_treat))

# z-score
z = (conv_treat - conv_control) / se

# two-tailed p-value
p_value = 2 * (1 - stats.norm.cdf(abs(z)))

print(f"z-score: {z:.4f}")
print(f"p-value: {p_value:.4f}")

alpha = 0.05
if p_value < alpha:
    print("Result: Statistically significant difference. Reject H0.")
else:
    print("Result: Not statistically significant. Fail to reject H0.")



z-score: 14.4828
p-value: 0.0000
Result: Statistically significant difference. Reject H0.


In [ ]:
z_crit = stats.norm.ppf(1 - alpha/2)  # ~1.96 for 95%

se_diff = np.sqrt(
    conv_control * (1 - conv_control) / n_control +
    conv_treat * (1 - conv_treat) / n_treat
)

ci_lower = (conv_treat - conv_control) - z_crit * se_diff
ci_upper = (conv_treat - conv_control) + z_crit * se_diff

print(f"95% CI for (Treatment - Control): [{ci_lower:.4f}, {ci_upper:.4f}]")


95% CI for (Treatment - Control): [0.0406, 0.0527]


In [ ]:
platform_summary = (
    session_df
    .groupby(["platform", "group"])["converted"]
    .agg(["count", "mean"])
    .reset_index()
    .rename(columns={"count": "sessions", "mean": "conversion_rate"})
)

print(platform_summary)


  platform      group  sessions  conversion_rate
0  Android    control     23021         0.838756
1  Android  treatment     12285         0.884900
2      iOS    control     11119         0.841083
3      iOS  treatment      5993         0.888703
